In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

In [2]:
df = pd.read_csv("../data/processed/features_crudop_logs.csv")

df.head()

,success_vol,success_rt_avg,fail_vol,fail_rt_avg,hour,hour_sin,hour_cos,operation_DELETE,operation_GET,operation_POST,operation_PUT,total_vol,fail_ratio,latency_gap,vol_per_latency
0,78996,12.441,418,2.383,0,0.000000,1.000000,False,True,False,False,79414,0.005263,10.058,5908.340153
1,9558,32.689,1554,10.685,0,0.000000,1.000000,False,False,True,False,11112,0.139836,22.004,329.840601
2,15498,34.332,2913,19.612,0,0.000000,1.000000,False,False,False,True,18411,0.158212,14.720,521.085701
3,6952,23.306,1762,12.749,0,0.000000,1.000000,True,False,False,False,8714,0.202180,10.557,358.512301
4,79648,7.007,411,3.606,1,0.258819,0.965926,False,True,False,False,80059,0.005134,3.401,9998.626202


In [3]:
ALL_FEATURES = [
    "success_vol",
    "fail_vol",
    "success_rt_avg",
    "fail_rt_avg",
    "hour_sin",
    "hour_cos",
    "hour"
] + [c for c in df.columns if c.startswith("operation_")]

In [4]:
def get_features_for_target(target, features):
    return [f for f in features if f != target]

In [5]:
TARGETS = [
    "success_vol",
    "fail_vol",
    "success_rt_avg",
    "fail_rt_avg"
]

In [6]:
def train_model(X_train, y_train, X_val, y_val):
    
    model = XGBRegressor(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        random_state=42,
        eval_metric="mae"
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    return model

In [7]:
models = {}
metrics = {}
feature_map = {}

for target in TARGETS:
    
    print(f"\nTraining model for: {target}")
    
    FEATURES = get_features_for_target(target, ALL_FEATURES)
    
    X = df[ALL_FEATURES]
    y = df[target]
    
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    model = train_model(X_train, y_train, X_val, y_val)
    
    preds = model.predict(X_val)
    mae = mean_absolute_error(y_val, preds)
    
    models[target] = model
    feature_map[target] = ALL_FEATURES
    metrics[target] = mae
    print(f"{target} features: {len(FEATURES)}")
    print(f"{target} MAE: {mae:.4f}")


Training model for: success_vol
success_vol features: 10
success_vol MAE: 84.0385

Training model for: fail_vol
fail_vol features: 10
fail_vol MAE: 1.0202

Training model for: success_rt_avg
success_rt_avg features: 10
success_rt_avg MAE: 0.0669

Training model for: fail_rt_avg
fail_rt_avg features: 10
fail_rt_avg MAE: 0.0252


In [8]:
import joblib
import os

bundle = {
    "models": models,
    "feature_map": feature_map,
    "targets": TARGETS
}

os.makedirs("models", exist_ok=True)

joblib.dump(bundle, "../models/v1/model_bundle.pkl")

print("Model saved successfully")

Model saved successfully
